# Plotting with matplotlib - animation

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## Exercise 1
The file 'annual_temperature.csv'contains weather data from the [met office website](https://www.metoffice.gov.uk/research/climate/maps-and-data/uk-and-regional-series).
We will explore this data visually using matplotlib. 

Plot the average annual temperature (column 'ann' in 'annual_temperature.csv') for each year in a line plot. 
On the same Axes, plot the five year rolling average temperature. 
That is, for each year, instead of plotting the annual temperature of that year only, plot the average temperature from that year and the past four. 
This is a way of smoothing the data to reduce noise and highlight trends.

Do the same for the average winter and summer temperature.
Create a figure with all three plots.
Add suitable labels and titles.

First import the data. 
I've used pandas but you could import using the CSV library instead.
There is one missing piece of data, it's a good idea to replace it with a numpy NaN at the beginning.

In [ ]:
df = pd.read_csv('annual_temperatures.csv')

df.head()

In [ ]:
# fix the bad winter value in the first row
df.loc[0,'win'] = np.nan
df['win'] = pd.to_numeric(df['win'])
df.head()

In [ ]:
df['5_year_average'] = df['ann'].rolling(5).mean()
df['5_year_summer'] = df['sum'].rolling(5).mean()
df['5_year_winter'] = df['win'].rolling(5).mean()

In [ ]:
df.head(10)

Plotting the data.

In [ ]:
fig, axs = plt.subplots(3,1,figsize=(10,10),sharex=True)

axs[0].plot('year', 'ann', data=df, lw=3, color='green', alpha=0.3, label='Average annual temperature')
axs[0].plot('year', '5_year_average', lw=1.5, data=df, color='green', label='5 year annual average')
axs[0].legend(loc='lower right')

axs[1].plot('year', 'win', data=df, lw=3, color='blue', alpha=0.3, label='Average winter temperature')
axs[1].plot('year', '5_year_winter', data=df, lw=1.5, color='blue', label='5 year winter average')
axs[1].legend(loc='lower right')

axs[2].plot('year', 'sum', data=df, lw=3, color='red', alpha=0.3, label='Average summer temperature')
axs[2].plot('year', '5_year_summer', data=df, lw=1.5, color='red', label='5 year summer average')
axs[2].legend(loc='lower right')

axs[2].set_xlim(1879, 2021)
axs[2].set_xticks(range(1880, 2021, 20))
axs[2].set_xlabel('year')

axs[0].set_title('Annual')
axs[1].set_title('Winter')
axs[2].set_title('Summer')

axs[0].set_ylabel('temp (C)')
axs[1].set_ylabel('temp (C)')
axs[2].set_ylabel('temp (C)')

fig.suptitle('Change in UK Temperature Over Time')

plt.tight_layout()


## Exercise 2
The following cells show how to animate one of the above plots using matplotlib's FuncAnimation 

The basic idea is to draw a sequence of plots onto the same axes, this gives the appearance of an animation. 
In the above example, instead of drawing the whole line, representing data across all years at once, we could draw the line only up to a given year.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4)) 

# take only the first 50 values for each plot
ax.plot(df['year'][:50], df['ann'][:50], lw=3, color='green', alpha=0.3, label='Average annual temperature')
ax.plot(df['year'][:50], df['5_year_average'][:50], lw=1.5, color='green', label='5 year annual average')
ax.legend(loc='lower right')

# keep the same limits on the x axis
ax.set_xlim(1879, 2021)
ax.set_xticks(range(1880, 2021, 20))
ax.set_xlabel('year')
ax.set_ylabel('temp (C)')

ax.set_title('Uk Annual Temperatures')

plt.tight_layout()

Now if we changed the part of the data we're selecting we can make a sequence of plots where the amount of line displayed gradually increases. 
Something like the following:

In [ ]:
num_years = 2021-1884
for i in range(num_years):
    ax.plot(df['year'][:i], df['ann'][:i], lw=3, color='green', alpha=0.3, label='Average annual temperature')
    ax.plot(df['year'][:i], df['5_year_average'][:i], lw=1.5, color='green', label='5 year annual average')

This is roughly what we will do except that a) instead of a loop we'll make a function which can be called by FuncAnimation (and it will be FuncAnimation that updates the value of i), b) we can set things like the color and label initially, since these will be the same for every value of i. c) instead of using `plot()` we will use `set_data()` which just updates the data that matplotlib plots and assumes the same type of plot is being used as before.

In [ ]:
from matplotlib.animation import FuncAnimation

We need to change the backend of matplotlib so that it can support animations in the notebook environment. 

In [ ]:
%matplotlib notebook

In [ ]:
# set up the figure and axes
fig, ax = plt.subplots(figsize=(6,4)) 

ax.set_xlim(1879, 2021)
ax.set_ylim(6.5,10)
ax.set_xticks(range(1880, 2021, 20))
ax.set_xlabel('year')
ax.set_title('Annual Temperatures')
ax.set_ylabel('temp')

# range of x axis
num_years = len(df)+1

# an initial plot, store the artists in variables so they can be updated later
line1, = ax.plot([],[])
line2, = ax.plot([],[])

# this function is run at the start of the animation
# set basic properties of the lines, create legend
def init():
    line1.set_linewidth(3)
    line1.set_color('green')
    line1.set_alpha(0.3)
    line1.set_label('Average annual temperature')
    line2.set_linewidth(1.5)
    line2.set_color('green')
    line2.set_label('5 year annual average')
    ax.legend(loc='lower right')
    return line1, line2, ax

# update the section of data being displayed
def animate(i):
    line1.set_data(df['year'][:i], df['ann'][:i])  
    line2.set_data(df['year'][:i], df['5_year_average'][:i])
    return line1, line2

ani = FuncAnimation(
    fig=fig, 
    init_func=init, 
    func=animate, 
    frames=range(num_years), 
    interval=200, 
    repeat=False #True
)

plt.tight_layout()
plt.show()

And if you like the animation, you can save it as a gif.

In [ ]:
ani.save('temps2.gif', writer='pillow')

## Exercise 3

Using the annual_temperatures file 
- plot the temperature of each month in a given year as a line. 
- write a function to draw this plot for any year. 
- make an animated plot that shows the plot for every year in order. 
- add a second animated plot showing the annual average temperature across the years which should develop in time with the plot showing the detailed information about each year.


In [ ]:
df2 = pd.read_csv('annual_temperatures.csv')

# fix the bad winter value in the first row
df2.loc[0,'win'] = np.nan
df2['win'] = pd.to_numeric(df2['win'])
df2

Save the annual averages data for later

In [ ]:
df3=df2[['year','ann']]
df3.head()

Transpose the data to make it easier to work with.

In [ ]:
df2.index=df2['year']
df2

In [ ]:
df2 = df2.drop('year', axis=1)
df2.head()

In [ ]:
df2 = df2.T
df2

Get rid of the columns we don't need and reset to a numerical index.

In [ ]:
df2 = df2.drop(['win', 'sum', 'spr', 'aut', 'ann'])
df2

In [ ]:
df2 = df2.reset_index()
df2

And let's make the column names a bit more sensible

In [ ]:
df2.columns.name = ''
df2 = df2.rename(columns = {'index': 'month'})

In [ ]:
df2

Quick check to see what to set as the upper / lower value for the y axis

In [ ]:
df2.max()[1:].max()

In [ ]:
df2.min()[1:].min()

Plot a single year

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

ax.set_ylim(-3, 20)
ax.set_xlabel('Month'),
ax.set_ylabel('Temperature')
ax.set_title('Annual Temperatures')
ax.plot(df2['month'], df2[1990])

plt.show()

To plot every year in turn we can simply change which column of the dataframe is being selected. 

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.set_ylim(-3,20)
ax.set_xlabel('Month')
ax.set_ylabel('Temperature')
ax.set_title(f'Temperature (°C) for ')

line, = ax.plot(df2['month'],[-5]*12)

def init():
    line.set_linewidth(3)
    line.set_color('green')
    line.set_alpha(0.3)
    return line

def animate(year):
    ax.set_title(f'Temperature (°C) for {year}')
    line.set_data(df2['month'], df2[year])
    return line
    
ani = FuncAnimation(
    fig=fig,
    init_func=init, 
    func=animate,
    frames=range(1884,2021),
    interval=20,
    repeat=False
)

fig.tight_layout()
fig.show()

Now to add a second plot:

In [ ]:
num_years = 2021-1884

fig, axs = plt.subplots(2,1,figsize=(7,4))

axs[0].set_ylim(6,10)
axs[1].set_ylim(-3,20)

axs[1].set_xlabel('Month')
axs[1].set_ylabel('Temperature')
axs[0].set_ylabel('Temperature')
axs[0].set_xlim(1879, 2021)
axs[0].set_xticks(range(1880, 2021, 20))

line1, = axs[1].plot(df2['month'],[-5]*12)
line2, = axs[0].plot([],[])

def init():
    line1.set_linewidth(3)
    line1.set_color('green')
    line1.set_alpha(0.3)
    line2.set_linewidth(3)
    line2.set_color('blue')
    line2.set_alpha(0.3)   
    return line1, line2

def animate(i):
    year = 1884+i
    axs[0].set_title(f'Average Temperatures (°C) 1884-{year}')
    
    line1.set_data(df2['month'], df2[year])
    line2.set_data(df3['year'][:i], df3['ann'][:i])
    
ani = FuncAnimation(
    fig=fig,
    init_func=init,
    func=animate, 
    frames=range(num_years), 
    interval=200, 
    repeat=False)

fig.tight_layout()
fig.show()

In [ ]:
ani.save('temps.gif', writer='pillow')

## Bonus
A simple example with a scatter graph.

In [ ]:
df_temp = pd.read_csv('annual_temperatures.csv')
df_sun = pd.read_csv('annual_sunshine_hours.csv')

df_temp.rename(columns={'may': 'may_temp'}, inplace=True)
df_sun.rename(columns={'may': 'may_sun'}, inplace=True)

df_comb = pd.merge(df_temp[['year','may_temp']], df_sun[['year','may_sun']], on='year', how='inner')

In [ ]:
df_comb

In [ ]:
df_comb[['may_sun', 'may_temp']][:50]

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.set_xlim(100, 280)
ax.set_ylim(6,14)
ax.set_ylabel('temperature')
ax.set_xlabel('sunshine hours')
ax.set_title('Temperature vs Sunshine in May')

# scat = ax.scatter(df_comb['may_sun'], df_comb['may_temp'])
scat = ax.scatter([],[])

def animate(i):
    year = df_comb['year'][i]
    ax.set_title(f'Temperature vs Sunshine in May 1919-{year}')
    scat.set_offsets(df_comb[['may_sun', 'may_temp']][:i])

ani = FuncAnimation(
    fig=fig,
    func=animate,
    frames=range(102),
    interval=100, 
    repeat=False
    
)

## For more ideas of how matplotlib can help with animations, read the [documentation](https://matplotlib.org/stable/api/animation_api.html)